# 🏆 강화학습 벽돌깨기 대회 — 내 두뇌를 학습시켜 올리기

**AI CITY BUILDERS** · 대회 순위표: **aicitybuilders.com/contest?c=brick**

1. 사이트와 **한 글자도 다르지 않은** 미니 벽돌깨기를 불러옵니다.
2. DQN 으로 **1~3분** 학습시킵니다. (GPU 필요 없음)
3. 가장 잘한 두뇌를 **brain.json** 으로 내려받아 대회 페이지에 올립니다.
4. 서버가 **그 두뇌로 게임 5판을 직접 플레이** 해서 점수를 매깁니다. (점수를 적어 내는 게 아니라, 조작이 안 됩니다)

**점수** = 5판 동안 깬 벽돌 수의 평균 (판마다 60개 만점, 3,000걸음 제한)

| 두뇌 | 점수 |
|---|---|
| 가만히 있기 | 4.4 |
| 공만 졸졸 따라가기 (손으로 짠 규칙) | **26.6 ← 기본 두뇌** |
| DQN 1분 학습 (이 코랩 그대로) | 30~40 |
| 만점 | 60 |

공만 따라가면 공을 놓치지는 않지만, 가운데로만 받아서 공이 위아래로만 오갑니다. **판의 끝으로 받아 비스듬히 쳐 올리는 법** 을 배워야 점수가 오릅니다.

## 1. 미니 벽돌깨기 불러오기

In [ ]:
import random, time, json, collections, os
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, matplotlib
from IPython.display import HTML, display
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
글꼴경로 = next((p for p in ['/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf',
                            '/System/Library/Fonts/Supplemental/AppleGothic.ttf'] if os.path.exists(p)), None)
if 글꼴경로:
    matplotlib.font_manager.fontManager.addfont(글꼴경로)
    plt.rcParams['font.family'] = matplotlib.font_manager.FontProperties(fname=글꼴경로).get_name()
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 미니 벽돌깨기 — 사이트(aicitybuilders.com/dqn)와 한 글자도 다르지 않은 규칙.
# ⚠️ 이 칸은 고치지 마세요. 고치면 코랩 점수와 대회 점수가 달라집니다.
# 삼각함수 없이 사칙연산과 sqrt 만 쓴다 → 사이트 채점과 점수가 똑같이 나온다.
import math

W, H = 300, 400
COLS, ROWS, BW, BH, TOP = 10, 6, 30, 12, 60
PW, PY, PSPEED = 50, 370, 6
SPEED, LIVES, MAX_STEPS = 4, 3, 3000
LAUNCH = [-2, -1, 0.5, 1.5, 2.5, -2.5, 1, -0.5]
ACTIONS = ['가만히', '왼쪽', '오른쪽']
SEEDS = [0, 1, 2, 3, 4]


class Game:
    def __init__(self, seed=0):
        self.seed = seed; self.px = W / 2; self.bx = 0.0; self.by = 0.0; self.vx = 0.0; self.vy = 0.0
        self.bricks = [1] * (COLS * ROWS); self.left = COLS * ROWS
        self.lives = LIVES; self.score = 0; self.steps = 0; self.launches = 0; self.done = False
        self._launch()

    def _launch(self):
        vx = LAUNCH[(self.seed * 3 + self.launches) % len(LAUNCH)]
        self.launches += 1
        self.bx = self.px; self.by = PY - 10
        self.vx = vx; self.vy = -math.sqrt(SPEED * SPEED - vx * vx)

    def state(self):
        return [self.px / W, self.bx / W, self.by / H, self.vx / SPEED, self.vy / SPEED,
                (self.bx - self.px) / W, self.left / (COLS * ROWS), self.lives / LIVES]

    def step(self, action):
        if self.done: return 0
        reward = 0
        if action == 1: self.px -= PSPEED
        elif action == 2: self.px += PSPEED
        if self.px < PW / 2: self.px = PW / 2
        if self.px > W - PW / 2: self.px = W - PW / 2

        self.bx += self.vx; self.by += self.vy
        if self.bx < 0: self.bx = -self.bx; self.vx = -self.vx
        if self.bx > W: self.bx = 2 * W - self.bx; self.vx = -self.vx
        if self.by < 0: self.by = -self.by; self.vy = -self.vy

        if TOP <= self.by < TOP + ROWS * BH:
            c = math.floor(self.bx / BW); r = math.floor((self.by - TOP) / BH)
            i = r * COLS + c
            if 0 <= c < COLS and self.bricks[i] == 1:
                self.bricks[i] = 0; self.left -= 1; self.score += 1; reward += 1
                self.vy = -self.vy

        if self.vy > 0 and PY <= self.by <= PY + 8 and self.px - PW / 2 - 4 <= self.bx <= self.px + PW / 2 + 4:
            off = (self.bx - self.px) / (PW / 2)
            if off < -1: off = -1
            if off > 1: off = 1
            self.vx = off * 3
            self.vy = -math.sqrt(SPEED * SPEED - self.vx * self.vx)
            self.by = PY

        if self.by > H:
            self.lives -= 1; reward -= 1
            if self.lives <= 0: self.done = True
            else: self._launch()
        if self.left == 0: self.done = True
        self.steps += 1
        if self.steps >= MAX_STEPS: self.done = True
        return reward


def q_values(brain, x):
    """사이트와 같은 순서로 더한다 (b 먼저, 그다음 입력 순서대로)."""
    h = x
    n = len(brain['layers'])
    for k, L in enumerate(brain['layers']):
        out = []
        for j in range(len(L['b'])):
            s = L['b'][j]
            row = L['W'][j]
            for i in range(len(row)):
                s += row[i] * h[i]
            out.append(0 if (k < n - 1 and s < 0) else s)
        h = out
    return h


def argmax(a):
    best = 0
    for i in range(1, len(a)):
        if a[i] > a[best]: best = i
    return best


def evaluate(brain, seeds=SEEDS):
    scores = []
    for s in seeds:
        g = Game(s)
        while not g.done:
            g.step(argmax(q_values(brain, g.state())))
        scores.append(g.score)
    return scores, round(sum(scores) / len(scores) * 10) / 10

print('✅ 미니 벽돌깨기 준비 · 상태', len(Game().state()), '개 · 행동', ACTIONS)

## 2. 상태 · 행동 · 보상 들여다보기

이번 게임의 **상태는 숫자 8개** 입니다 (화면 전체가 아니라 요약본).

| 번호 | 상태 | 뜻 |
|---|---|---|
| 0 | 판 위치 | 0 왼쪽 끝 ~ 1 오른쪽 끝 |
| 1, 2 | 공 x, 공 y | 공의 위치 |
| 3, 4 | 공 가로·세로 속도 | 공이 어디로 가나 (세로 − 는 위로) |
| 5 | 공 − 판 | + 면 공이 판보다 오른쪽 |
| 6 | 남은 벽돌 | 1 → 0 |
| 7 | 남은 목숨 | 1 → 0 |

보상: 벽돌 하나 **+1** · 공을 놓치면 **−1**

In [ ]:
이름 = ['판', '공x', '공y', '가로', '세로', '공-판', '벽돌', '목숨']
g = Game(0)
print(f"{'걸음':>4} | {'행동':<4} | {'보상':>4} | " + ' '.join(f'{n:>6}' for n in 이름))
print('-' * 84)
for 걸음 in range(1, 301):
    s = g.state()
    a = random.randrange(3)                         # 아직은 아무거나
    r = g.step(a)
    if 걸음 <= 10 or r != 0:
        print(f"{걸음:>4} | {ACTIONS[a]:<4} | {r:>+4} | " + ' '.join(f'{v:>6.2f}' for v in s)
              + ('  ⭐ 벽돌!' if r > 0 else '  💥 공 놓침' if r < 0 else ''))
    if g.done: break
print(f'\n🎲 무작위: {g.steps}걸음 · 벽돌 {g.score}개 · 남은 목숨 {g.lives}')

## 3. 게임 화면으로 보기 (영상)

In [ ]:
from PIL import Image, ImageDraw
import imageio, base64
색 = [(248, 113, 113), (251, 146, 60), (250, 204, 21), (74, 222, 128), (56, 189, 248), (167, 139, 250)]

def 그리기(g, Q값=None, a=None):
    im = Image.new('RGB', (W + 170, H), (14, 18, 28)); d = ImageDraw.Draw(im)
    for i, b in enumerate(g.bricks):
        if b: r, c = divmod(i, COLS); d.rectangle([c*BW+1, TOP+r*BH+1, c*BW+BW-2, TOP+r*BH+BH-2], fill=색[r])
    d.rectangle([g.px-PW/2, PY, g.px+PW/2, PY+6], fill=(226, 232, 240))
    d.ellipse([g.bx-4, g.by-4, g.bx+4, g.by+4], fill=(255, 255, 255))
    d.line([W, 0, W, H], fill=(51, 65, 85))
    d.text((W+12, 12), f'벽돌 {g.score}', fill=(255, 214, 102), font=_f(20)); d.text((W+12, 40), f'목숨 {g.lives} · {g.steps}걸음', fill=(160, 170, 190), font=_f(13))
    if Q값 is not None:
        lo, hi = min(min(Q값), 0), max(max(Q값), 0.01)
        for k in range(3):
            y = 90 + k*50; L = int(140 * (Q값[k]-lo)/(hi-lo+1e-9))
            d.text((W+12, y), f'{ACTIONS[k]} {Q값[k]:.2f}', fill=(230, 235, 245) if k == a else (140, 150, 170), font=_f(14))
            d.rectangle([W+12, y+20, W+12+max(L, 2), y+34], fill=(56, 189, 248) if k == a else (71, 85, 105))
    return np.array(im)

def _f(n):
    from PIL import ImageFont
    try: return ImageFont.truetype(글꼴경로, n)
    except Exception: return ImageFont.load_default()

def 영상보기(화면들, fps=60):
    이름 = f'mini_{int(time.time()*1000)}.mp4'; imageio.mimsave(이름, 화면들, fps=fps, macro_block_size=1)
    display(HTML(f'<video width="470" controls autoplay loop muted><source src="data:video/mp4;base64,{base64.b64encode(open(이름,"rb").read()).decode()}" type="video/mp4"></video>'))

def 플레이영상(두뇌=None, 씨앗=0, 최대=1500, 건너뛰기=2):
    g = Game(씨앗); 화면들 = []
    while not g.done and g.steps < 최대:
        if 두뇌 is None: Q값, a = None, random.randrange(3)
        else: Q값 = q_values(두뇌, g.state()); a = argmax(Q값)
        g.step(a)
        if g.steps % 건너뛰기 == 0: 화면들.append(그리기(g, Q값, a))
    영상보기(화면들); return g.score

print('🎲 무작위 두뇌 플레이 → 벽돌', 플레이영상(None))

## 4. DQN 두뇌 만들기와 학습 🚀

아타리 때와 **똑같은 DQN** 입니다. 달라진 건 두뇌의 눈뿐 (화면 대신 숫자 8개 → 작은 두뇌 8 → 64 → 64 → 3).

2만 걸음마다 **대회와 똑같이 5판 채점** 해서, 가장 잘한 두뇌를 `최고_두뇌` 에 따로 챙겨 둡니다.

> 💡 DQN 은 잘하다가 갑자기 무너지기도 합니다. 그래서 **가장 좋았던 순간의 두뇌** 를 저장해 두는 게 중요합니다.

In [ ]:
총걸음 = 200_000     # 약 1분. 더 해 보고 싶으면 늘리기
은닉 = 64            # 두뇌 크기 (최대 256)
학습률 = 5e-4
감마 = 0.99          # 미래 보상을 얼마나 중요하게 볼까
탐험기간 = 0.4       # 전체의 몇 %동안 ε 을 1 → 0.02 로 줄일까
씨앗 = 0

random.seed(씨앗); np.random.seed(씨앗); torch.manual_seed(씨앗); torch.set_num_threads(1)
def 두뇌만들기(): return nn.Sequential(nn.Linear(8, 은닉), nn.ReLU(), nn.Linear(은닉, 은닉), nn.ReLU(), nn.Linear(은닉, 3))
q = 두뇌만들기(); 목표 = 두뇌만들기(); 목표.load_state_dict(q.state_dict()); opt = torch.optim.Adam(q.parameters(), lr=학습률)

N = 50_000                                                          # ② 기억 (리플레이 버퍼)
S = np.zeros((N, 8), np.float32); A = np.zeros(N, np.int64); R = np.zeros(N, np.float32)
S2 = np.zeros((N, 8), np.float32); D = np.zeros(N, np.float32); k = 0; 찬 = 0

def 내보내기(net):                                                   # 대회 사이트가 읽는 모양
    return {'layers': [{'W': m.weight.detach().double().tolist(), 'b': m.bias.detach().double().tolist()}
                       for m in net if isinstance(m, nn.Linear)]}

g = Game(random.randrange(1000)); s = g.state(); t0 = time.time()
최근 = collections.deque(maxlen=20); 최고점 = -1; 최고_두뇌 = None; 기록 = []
for 걸음 in range(1, 총걸음 + 1):
    eps = max(0.02, 1 - 걸음 / (총걸음 * 탐험기간))                    # ① 탐험
    if random.random() < eps: a = random.randrange(3)
    else:
        with torch.no_grad(): a = int(q(torch.tensor(s)).argmax())
    r = g.step(a); s2 = g.state()
    S[k] = s; A[k] = a; R[k] = r; S2[k] = s2; D[k] = float(g.done or r < 0); k = (k + 1) % N; 찬 = min(찬 + 1, N)
    s = s2
    if g.done: 최근.append(g.score); g = Game(random.randrange(1000)); s = g.state()
    if 걸음 > 2000:                                                  # 복습
        i = np.random.randint(0, 찬, 64)
        with torch.no_grad(): y = torch.tensor(R[i]) + 감마 * 목표(torch.tensor(S2[i])).max(1).values * (1 - torch.tensor(D[i]))
        p = q(torch.tensor(S[i])).gather(1, torch.tensor(A[i]).unsqueeze(1)).squeeze(1)
        loss = F.smooth_l1_loss(p, y); opt.zero_grad(); loss.backward(); opt.step()
    if 걸음 % 1000 == 0: 목표.load_state_dict(q.state_dict())         # ③ 목표 두뇌
    if 걸음 % 20_000 == 0:
        뇌 = 내보내기(q); 판들, 평균 = evaluate(뇌)                       # 대회와 똑같은 채점
        새기록 = 평균 > 최고점
        if 새기록: 최고점, 최고_두뇌 = 평균, 뇌
        with torch.no_grad(): Qs = q(torch.tensor(Game(0).state())).tolist()
        기록.append((걸음, np.mean(최근) if 최근 else 0, 평균))
        print(f"[{걸음:>7}걸음 | {time.time()-t0:4.0f}초 | ε {eps:.2f}] 연습 평균 {np.mean(최근) if 최근 else 0:5.1f} | 대회 채점 {평균:5.1f} {판들}"
              + ('  🏅 최고 기록!' if 새기록 else ''))
        print(f"      첫 화면 상태에서 Q값 → " + ' · '.join(f'{ACTIONS[j]} {Qs[j]:+.2f}' for j in range(3)))
print(f'\n✅ 끝! 최고 대회 점수 {최고점} (기본 두뇌 26.6)')

In [ ]:
x = [b[0] for b in 기록]
plt.figure(figsize=(9, 3.5))
plt.plot(x, [b[1] for b in 기록], label='연습 판 평균 (ε 탐험 포함)')
plt.plot(x, [b[2] for b in 기록], 'o-', lw=2.5, label='대회 채점 (5판)')
plt.axhline(26.6, ls='--', c='gray', label='기본 두뇌 26.6'); plt.axhline(60, ls=':', c='gold', label='만점 60')
plt.xlabel('걸음'); plt.ylabel('벽돌'); plt.title('내 두뇌가 배운 기록'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 5. 내 최고 두뇌가 하는 걸 보자

In [ ]:
print('🧠 내 최고 두뇌 → 벽돌', 플레이영상(최고_두뇌, 씨앗=1))

## 6. 대회에 내기 📤

아래 칸을 돌리면 **brain.json** 이 내려받아집니다.
👉 **aicitybuilders.com/contest?c=brick** 에 들어가서 그 파일을 끌어다 놓으면 끝.

하루 5번 · 전체 30번까지 낼 수 있습니다. 4번 칸의 숫자(총걸음·은닉·학습률·감마·탐험기간·씨앗)를 바꿔 가며 기록을 올려 보세요.

In [ ]:
판들, 평균 = evaluate(최고_두뇌)
print(f'📋 대회와 똑같이 채점: {판들} → 평균 {평균}개  (사이트에서도 이 점수가 나옵니다)')
json.dump(최고_두뇌, open('brain.json', 'w'))
print('💾 brain.json 저장 ·', round(os.path.getsize('brain.json') / 1024), 'KB')
try:
    from google.colab import files; files.download('brain.json')
except ImportError:
    print('코랩이 아니면 왼쪽 파일 목록에서 brain.json 을 받으세요.')